In [4]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import os
from dotenv import load_dotenv

load_dotenv()

url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"), 
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(url)

with engine.connect() as conn:
    print("Connected successfully!")

Connected successfully!


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
query = """
SELECT *
FROM analytics.product_affinity_features
"""

product_affinity_features_df = pd.read_sql(query, engine)

product_affinity_features_df.shape

(3492837, 6)

In [7]:
affinity_df = product_affinity_features_df

In [8]:
product_affinity_features_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3492837 entries, 0 to 3492836
Data columns (total 6 columns):
 #   Column             Dtype  
---  ------             -----  
 0   product_id_a       int64  
 1   product_id_b       int64  
 2   co_purchase_count  int64  
 3   support            float64
 4   confidence         float64
 5   lift               float64
dtypes: float64(3), int64(3)
memory usage: 159.9 MB


In [9]:
affinity_df[[
    "co_purchase_count",
    "support",
    "confidence",
    "lift"
]].describe(
    percentiles=[0.25,0.50,0.75,0.90,0.95,0.99]
)

,co_purchase_count,support,confidence,lift
count,3.492837e+06,3.492837e+06,3.492837e+06,3.492837e+06
mean,4.644398e+01,1.444608e-05,1.423784e-02,2.285937e+01
std,2.247370e+02,6.990545e-05,2.977316e-02,4.713329e+02
min,1.000000e+01,3.110000e-06,2.116000e-05,8.976840e-03
25%,1.300000e+01,4.040000e-06,1.892960e-03,1.471497e+00
50%,1.800000e+01,5.600000e-06,5.145800e-03,2.357470e+00
75%,3.400000e+01,1.058000e-05,1.377727e-02,4.673198e+00
90%,7.500000e+01,2.333000e-05,3.379953e-02,1.207732e+01
95%,1.330000e+02,4.137000e-05,5.555556e-02,2.642301e+01
99%,4.610000e+02,1.434000e-04,1.483247e-01,2.169848e+02


In [10]:
affinity_df.sort_values(
    "lift",
    ascending=False
)[[
    "product_id_a",
    "product_id_b",
    "co_purchase_count",
    "support",
    "confidence",
    "lift"
]].head(20)

,product_id_a,product_id_b,co_purchase_count,support,confidence,lift
765524,36997,40669,15,0.000005,0.937500,167441.354167
885255,43603,46262,16,0.000005,0.842105,128917.253133
3174356,3748,8270,12,0.000004,0.571429,122471.390476
1083106,2352,2686,10,0.000003,0.588235,105061.241830
610787,8616,38756,11,0.000003,0.578947,103402.380117
2521486,5304,43527,10,0.000003,0.588235,94555.117647
342940,12783,43579,11,0.000003,0.611111,81860.217593
423525,26465,35928,42,0.000013,1.000000,76544.619048
539867,2345,37293,11,0.000003,0.478261,73216.592133
1400825,14777,43553,13,0.000004,0.295455,73065.318182


In [11]:
affinity_df.sort_values(
    "co_purchase_count",
    ascending=False
)[[
    "product_id_a",
    "product_id_b",
    "co_purchase_count",
    "support",
    "confidence",
    "lift"
]].head(20)

,product_id_a,product_id_b,co_purchase_count,support,confidence,lift
1095380,13176,47209,62341,0.019391,0.164293,2.472945
1104555,13176,21137,61628,0.019170,0.162414,1.972702
2572745,21137,24852,56156,0.017468,0.212163,1.443353
2229880,24852,47766,53395,0.016609,0.112990,2.054395
2972592,21903,24852,51395,0.015987,0.212445,1.445272
1104279,13176,21903,50372,0.015668,0.132750,1.764107
570907,16797,24852,41232,0.012825,0.288434,1.962229
2229952,24852,47626,40880,0.012716,0.086507,1.821783
2566266,21137,47209,40794,0.012689,0.154124,2.319880
1102149,13176,27966,40503,0.012599,0.106741,2.503775


In [12]:
basket_features = (
    affinity_df
    .groupby("product_id_a")
    .agg(
        relationship_count=("product_id_b", "count"),
        total_co_purchase_count=("co_purchase_count", "sum"),
        avg_confidence=("confidence", "mean"),
        avg_lift=("lift", "mean"),
        p95_lift=("lift", lambda x: x.quantile(0.95)),
        avg_support=("support", "mean")
    )
)

In [13]:
basket_features.describe()

,relationship_count,total_co_purchase_count,avg_confidence,avg_lift,p95_lift,avg_support
count,25225.000000,2.522500e+04,25225.000000,25225.000000,25225.000000,25225.000000
mean,138.467275,6.430971e+03,0.098545,578.171511,1028.850817,0.000006
std,442.666058,4.980355e+04,0.101175,3601.645380,4498.370857,0.000004
min,1.000000,1.000000e+01,0.000185,0.406511,0.406511,0.000003
25%,3.000000,4.600000e+01,0.027535,4.465280,10.055108,0.000004
50%,15.000000,2.450000e+02,0.064753,15.277390,44.909782,0.000005
75%,80.000000,1.671000e+03,0.135802,87.361040,284.955233,0.000007
max,12888.000000,3.646441e+06,1.000000,167441.354167,167441.354167,0.000090


In [14]:
basket_features

,relationship_count,total_co_purchase_count,avg_confidence,avg_lift,p95_lift,avg_support
product_id_a,,,,,,
1,321,11470,0.019294,20.288234,61.239105,0.000011
2,2,23,0.127778,1.350174,1.463912,0.000004
3,47,821,0.063062,89.237014,608.836130,0.000005
4,46,833,0.055042,363.264508,1164.130995,0.000006
7,1,12,0.400000,1847.628736,1847.628736,0.000004
...,...,...,...,...,...,...
49667,1,44,0.027312,0.902280,0.902280,0.000014
49670,1,11,0.048035,1.586870,1.586870,0.000003
49678,1,13,0.034483,1.139164,1.139164,0.000004


In [15]:
basket_a = (
    affinity_df
    .groupby("product_id_a")
    .agg(
        relationship_count=("product_id_b", "count"),
        total_co_purchase_count=("co_purchase_count", "sum"),
        avg_confidence=("confidence", "mean"),
        avg_lift=("lift", "mean"),
        p95_lift=("lift", lambda x: x.quantile(0.95))
    )
    .reset_index()
    .rename(columns={"product_id_a": "product_id"})
)

In [16]:
basket_b = (
    affinity_df
    .groupby("product_id_b")
    .agg(
        relationship_count=("product_id_a", "count"),
        total_co_purchase_count=("co_purchase_count", "sum"),
        avg_confidence=("confidence", "mean"),
        avg_lift=("lift", "mean"),
        p95_lift=("lift", lambda x: x.quantile(0.95))
    )
    .reset_index()
    .rename(columns={"product_id_b": "product_id"})
)

In [17]:
basket_features = pd.concat(
    [basket_a, basket_b],
    ignore_index=True
)

In [18]:
basket_features = (
    basket_features
    .groupby("product_id")
    .agg(
        relationship_count=("relationship_count", "sum"),
        total_co_purchase_count=("total_co_purchase_count", "sum"),
        avg_confidence=("avg_confidence", "mean"),
        avg_lift=("avg_lift", "mean"),
        p95_lift=("p95_lift", "mean")
    )
    .reset_index()
)

In [19]:
basket_features.shape

(28524, 6)

In [20]:
basket_features.describe()

,product_id,relationship_count,total_co_purchase_count,avg_confidence,avg_lift,p95_lift
count,28524.000000,28524.000000,2.852400e+04,28524.000000,28524.000000,28524.000000
mean,24819.900855,244.905133,1.137437e+04,0.066091,745.584999,1243.822032
std,14340.515376,751.618007,8.489495e+04,0.089451,4331.765236,5163.583341
min,1.000000,1.000000,1.000000e+01,0.000021,0.390979,0.390979
25%,12523.250000,5.000000,6.300000e+01,0.013043,5.102006,11.874918
50%,24756.000000,24.000000,3.810000e+02,0.032189,19.701232,60.964841
75%,37235.000000,142.000000,2.845250e+03,0.078362,130.647629,432.595341
max,49688.000000,23516.000000,6.113628e+06,1.000000,167441.354167,167441.354167


In [21]:
basket_features.sort_values(
    "relationship_count",
    ascending=False
).head(20)

,product_id,relationship_count,total_co_purchase_count,avg_confidence,avg_lift,p95_lift
14313,24852,23516,6113628,0.097744,1.325958,2.325560
7526,13176,17489,4762878,0.077181,1.301982,2.454161
9651,16797,14474,1799241,0.034705,1.544687,3.287823
12148,21137,14404,3695145,0.057679,1.378700,2.627094
27322,47626,14192,2219015,0.036831,1.549262,3.153402
27407,47766,13210,2453833,0.040694,1.478585,3.203948
15090,26209,13205,2132145,0.033832,1.523363,3.097677
27074,47209,12887,3071643,0.057323,1.686470,3.184029
12608,21903,12429,3320424,0.056581,1.485235,2.622380
16071,27966,12094,2021029,0.037304,1.702586,3.425773


In [22]:
basket_features["log_total_co_purchase_count"] = np.log1p(
    basket_features["total_co_purchase_count"]
)

In [24]:
basket_features[
    [
        "relationship_count",
        "total_co_purchase_count",
        "avg_confidence",
        "avg_lift",
        "p95_lift"
    ]
].corr()

,relationship_count,total_co_purchase_count,avg_confidence,avg_lift,p95_lift
relationship_count,1.000000,0.817557,-0.190105,-0.054353,-0.074175
total_co_purchase_count,0.817557,1.000000,-0.072671,-0.022743,-0.031477
avg_confidence,-0.190105,-0.072671,1.000000,0.453073,0.475096
avg_lift,-0.054353,-0.022743,0.453073,1.000000,0.946759
p95_lift,-0.074175,-0.031477,0.475096,0.946759,1.000000


In [27]:
basket_features["log_relationship_count"] = np.log1p(
    basket_features["relationship_count"]
)

basket_features["basket_breadth_score"] = (
    basket_features["log_relationship_count"]
      .rank(pct=True)
      * 100
)

basket_features["basket_strength_score"] = (
    basket_features["avg_confidence"]
      .rank(pct=True)
      * 100
)

basket_features["basket_influence_score"] = (
    0.7*basket_features["basket_breadth_score"]
    +
    0.3*basket_features["basket_strength_score"]
) / 2

In [28]:
basket_features.sort_values(
    "basket_influence_score",
    ascending=False
).head(20)

,product_id,relationship_count,total_co_purchase_count,avg_confidence,avg_lift,p95_lift,log_total_co_purchase_count,log_relationship_count,basket_breadth_score,basket_strength_score,basket_influence_score
14313,24852,23516,6113628,0.097744,1.325958,2.325560,15.626031,10.065479,100.000000,79.718833,46.957825
7526,13176,17489,4762878,0.077181,1.301982,2.454161,15.376363,9.769385,99.996494,74.659935,46.197763
12148,21137,14404,3695145,0.057679,1.378700,2.627094,15.122531,9.575331,99.989483,67.157481,45.069941
27074,47209,12887,3071643,0.057323,1.686470,3.184029,14.937723,9.464052,99.975459,66.947132,45.033481
12608,21903,12429,3320424,0.056581,1.485235,2.622380,15.015603,9.427868,99.971953,66.557986,44.973882
27407,47766,13210,2453833,0.040694,1.478585,3.203948,14.713162,9.488805,99.982471,56.871407,43.524576
16071,27966,12094,2021029,0.037304,1.702586,3.425773,14.519118,9.400547,99.968448,54.287617,43.132099
27322,47626,14192,2219015,0.036831,1.549262,3.153402,14.612574,9.560504,99.985977,53.933530,43.085121
9651,16797,14474,1799241,0.034705,1.544687,3.287823,14.402876,9.580178,99.992988,52.226195,42.831475
15090,26209,13205,2132145,0.033832,1.523363,3.097677,14.572640,9.488427,99.978965,51.479456,42.714556
